In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

# this is used for our langsmith tracking
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_PROJECT'] = os.getenv('LANGCHAIN_PROJECT')

os.environ['LANGCHAIN_TRACING_V2'] = 'true'

In [19]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4o'
)
print(llm)

profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x000002CDAFC822F0> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002CDAFC82E00> root_client=<openai.OpenAI object at 0x000002CDAFC834C0> root_async_client=<openai.AsyncOpenAI object at 0x000002CDAFC83280> model_name='gpt-4o' model_kwargs={} openai_api_key=SecretStr('**********') stream_usage=True


In [2]:
# From the website we need to scrape the data we use beautifulsoup here
from langchain_community.document_loaders import WebBaseLoader


c:\Generative AI\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [8]:
loader = WebBaseLoader('https://en.wikipedia.org/wiki/Egyptian_pyramids')
loader

In [9]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Egyptian_pyramids', 'title': 'Egyptian pyramids - Wikipedia', 'language': 'en'}, page_content='\n\n\n\nEgyptian pyramids - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\nDonate Create account Log in\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nContents\nmove to sidebar\nhide\n\n\n\n\n(Top)\n\n\n\n\n\n1\nName\n\n\n\n\n\n\n\n\n2\nHistorical development\n\n\n\n\n\n\n\n

# now our data is loaded now we need to divide these into chunk bcuz every llm have a context size, after that we need to convert those to embeddings


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

final_chunks = splitter.split_documents(docs)
final_chunks

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Egyptian_pyramids', 'title': 'Egyptian pyramids - Wikipedia', 'language': 'en'}, page_content='Egyptian pyramids - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\nDonate Create account Log in\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nContents\nmove to sidebar\nhide\n\n\n\n\n(Top)\n\n\n\n\n\n1\nName\n\n\n\n\n\n\n\n\n2\nHistorical development\n\n\n\n\n\n\n\n\n3\nPyr

In [11]:
len(final_chunks)

49

# Now lets convert thse chunks into vectors

In [12]:
# we need to convert these into vectors becase every llm use something called cosine similarity to do the work.. to it to work we need to covnert these into numbers

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

In [ ]:
# We use this vectorstore db to save our embeddings
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(
    documents=final_chunks,
    embedding=embeddings
)

In [14]:
db

In [15]:
query = 'What is the largest Pyramid in Egypt'

results = db.similarity_search(query=query)
results[0].page_content

"The earliest known Egyptian pyramids are at Saqqara, west of Memphis. Step-pyramid-like structures, like Mastaba 3808 attributed to pharaoh Anedjib,[6] may predate the Pyramid of Djoser built c.\xa02630–2610\xa0BCE during the Third Dynasty.[7] This pyramid and its surrounding complex are generally considered to be the world's oldest monumental structures constructed of dressed masonry.[8]\nThe most famous Egyptian pyramids are those found at Giza, on the outskirts of Cairo. Several of the Giza pyramids are counted among the largest structures ever built.[9] The Pyramid of Khufu is the largest Egyptian pyramid and the last of the Seven Wonders of the Ancient World still in existence, despite being the oldest by about 2,000 years.[10]"

# lets create a retrieval chain

In [28]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context below.
If the answer isn't in the context, say you don't know.

<context>
{context}
</context>

Question: {input}
""")



document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt
)

document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template="\nAnswer the question using ONLY the context below.\nIf the answer isn't in the context, say you don't know.\n\n<context>\n{context}\n</context>\n\nQuestion: {input}\n"), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_mess

In [29]:
from langchain_core.documents import Document

In [30]:
document_chain.invoke({
    "input":"What is the largest Pyramid in Egypt",
    'context':[Document(page_content="The most famous Egyptian pyramids are those found at Giza, on the outskirts of Cairo. Several of the Giza pyramids are counted among the largest structures ever built. The Pyramid of Khufu is the largest Egyptian pyramid and the last of the Seven Wonders of the Ancient World still in existence, despite being the oldest by about 2,000 years.")]
})

'The largest Pyramid in Egypt is the Pyramid of Khufu.'

# Retriever

In [31]:
retriever = db.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002CDB0532EC0>, search_kwargs={})

In [32]:
from langchain_classic.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain
)

In [33]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002CDB0532EC0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template="\nAnswer the question using ONLY the context below.\nIf the answer isn't in the context, say you don't know.\n\n<context>\n{context

# Get the response from the llm

In [34]:
response = retrieval_chain.invoke({'input':'What is the largest Pyramid in Egypt'})
response['answer']

'The largest Egyptian pyramid is the Pyramid of Khufu.'